# Неделя 9 — Предобученный BERT: feature extraction

Цель: применить pretrained encoder без изменения его весов. В базовом BERT нет sentiment-головы. Здесь используется разрешённый вариантом задания feature extraction + LR, это не zero-shot классификация.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'nlp_project').exists() else Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import json
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown
from nlp_project.core import corpus, clean, tokens, metrics, SEED
df = corpus()
train, val, test = [df[df.split == s].copy() for s in ('train', 'validation', 'test')]


In [ ]:
import joblib
from transformers import AutoTokenizer, BertModel
from nlp_project.transformer import encode, seed_all
seed_all()
path = ROOT / 'models/pretrained_bert'
tokenizer = AutoTokenizer.from_pretrained(path, local_files_only=True)
encoder = BertModel.from_pretrained(path, local_files_only=True)
head = joblib.load(ROOT / 'models/bert_frozen_lr.joblib')
z = encode(encoder, tokenizer, test.text.tolist())
prob = head.predict_proba(z)[:,1]
display(pd.DataFrame({'text': test.text.values, 'label': test.label.values, 'prediction': (prob >= .5).astype(int), 'p_positive': prob}).head(15))
print(metrics(test.label, prob >= .5))
display(pd.read_csv(ROOT / 'reports/all_metrics.csv').query("model in ['tfidf_lr','bow_nb','bert_frozen_lr']"))


## Выводы

Замороженный BERT использует внешнее предобучение, но LR обучается на train. Это отделяет пользу представлений от fine-tuning. Сравнение нужно читать вместе с размером корпуса, параметрами pooling и неопределённостью малого test.